# Retrieval-Augmented Generation (RAG) Pipeline

This notebook demonstrates the complete RAG pipeline.

The pipeline is divided into two major phases:

1. **Document Ingestion and Indexing**
   - Read the document
   - Preprocess the extracted text
   - Create chunks
   - Generate embeddings
   - Store the embeddings in a vector store

2. **Query and Response Generation**
   - Receive a user query
   - Generate the query embedding
   - Retrieve relevant document chunks
   - Build a prompt using the query and retrieved context
   - Generate the final response using an LLM

# Phase 1: Document Ingestion and Indexing

In this phase, the source document is transformed into searchable vector representations.

The document passes through the following pipeline:

```PDF
↓
Text Extraction
↓
Text Preprocessing
↓
Chunking
↓
Embedding Generation
↓
Vector Store

## 1. PDF Reading and Markdown Extraction

The first step is to extract the content from the PDF document.

The document is initially converted into a markdown representation.

In [ ]:
import pymupdf
import pymupdf4llm
from src.preprocessing.ingestion_preprocessing import HeaderInfoBuilder
FILE_PATH ="./src/pdfs/logical_reasoning.pdf"

In [ ]:
header_builder = HeaderInfoBuilder()
hdr_info = header_builder.build(FILE_PATH)

In [ ]:
pymupdf4llm.use_layout(False)

doc = pymupdf.open(FILE_PATH)

markdown = pymupdf4llm.to_markdown(
    doc,
    show_progress=True,
    # use_ocr=False,
    hdr_info=hdr_info
)


## 2. Text Preprocessing

The raw text extracted from a PDF may contain structural noise such as:

- Repeated headers
- Repeated footers
- Page numbers
- Other boilerplate content

The purpose of this stage is to identify and remove repeated structural content while preserving the actual semantic content of the document.

The preprocessing pipeline uses similarity analysis and statistical patterns to identify boilerplate content dynamically rather than relying on document-specific hardcoded strings.

In [ ]:
"""
Markdown Preprocessing is pending and below code is written to preprocess the text file.
so it is commented out.
"""
# from src.preprocessing.cleaner import normalize_pdf
# from src.preprocessing.similarity import compute_line_similarity
# from src.preprocessing.boilerplate import identify_boilerplate, remove_boilerplate

In [ ]:
# # Phase 1: PDF Text normalization
# normalized_pdf = normalize_pdf(pdf)

# # # Phase 2: Statistical Analysis
# similar_line_analysis = compute_line_similarity(pdf=normalized_pdf, candidate_lines=3)

# # Phase 3: Boilerplate Identification
# candidate_boilerplates = identify_boilerplate(similarity_df=similar_line_analysis)

# # print(candidate_boilerplates)

# # # Phase 4: Boilerplate Removal
# boilerplate_config = (
#     candidate_boilerplates[
#         candidate_boilerplates["is_boilerplate"]
#     ][["region", "line"]]
# )

# filtered_pdf = remove_boilerplate(normalized_pdf, boilerplate_config=boilerplate_config)

# # Phase 5: Converting page-wise PDF into the one combined text
# text = "".join(filtered_pdf)

In [ ]:
# Markdown -> Tree
from src.preprocessing.markdown_parser import build_tree

markdown_tree = build_tree(text=markdown)

## 3. Text Chunking

The cleaned document is divided into smaller segments called chunks.

Chunking is necessary because the entire document cannot be efficiently embedded and retrieved as a single unit.

The objective of chunking is to create meaningful units of text that:

- Preserve sufficient semantic context
- Are small enough for efficient embedding and retrieval
- Can be independently retrieved in response to a user query

### Chunking — V1

The chunking process is performed in two layers.

#### 1. Semantic Chunking

First, we traverse the Markdown tree and create chunks based on the document's semantic hierarchy.

Each node that contains content becomes a chunk candidate.

For every chunk, we keep:

- `title`
- `content`
- `breadcrumb`

For parent nodes, the content also includes the node's subtopics.

The purpose of this layer is to preserve the **meaning and hierarchy** of the document.

> Semantic chunking answers: **"Where does the content naturally belong?"**

In [ ]:
from src.chunking.semantic_chunking import semantic_chunker

In [ ]:
chunks = semantic_chunker([markdown_tree])

In [ ]:
# Verify
chunks[2:]

#### 2. Recursive Text Splitter

The semantic chunks may still be too large for embedding.

Therefore, we pass the content of each semantic chunk through the `RecursiveCharacterTextSplitter`.

This layer only handles **size control**. It splits a large semantic chunk into smaller pieces while trying to preserve the local context.

> Recursive splitting answers: **"Is this semantic chunk small enough to process?"**

#### Pipeline

Markdown Tree  
↓  
**Semantic Chunking** → meaningful, hierarchy-aware chunks  
↓  
**Recursive Text Splitting** → size-controlled chunks  
↓  
Final chunks for embedding

In [ ]:
from src.chunking.chunker import text_splitter

In [ ]:
final_chunks = []
for chunk in chunks:
   splitted_chunks =  text_splitter(chunk['content'])

   final_chunks.append({
      'title': chunk['title'],
      'chunks': splitted_chunks,
      'breadcrumb': chunk['breadcrumb']
   })

In [ ]:
final_chunks[7:10]

## 4. Document Embedding Generation

Each text chunk is converted into a numerical vector representation called an embedding.

The embedding represents the semantic meaning of the chunk in a vector space.

Semantically similar chunks should have similar vector representations.

The result of this stage is:

Text Chunks
        ->
Embedding Model
        ->
Vector Representations

In [ ]:
from src.embeddings.embedder import chunk_encoder

In [ ]:
embeddings = chunk_encoder(chunks)

embeddings

## 5. Vector Store

The generated embeddings are stored in a vector index.

The vector store is responsible for:

- Storing document embeddings
- Maintaining the relationship between vectors and their original text chunks
- Searching for vectors that are most similar to a query embedding

In this project, FAISS is used as the vector search backend.

The document ingestion pipeline is now complete:

PDF
->
Text Extraction
->
Preprocessing
->
Chunking
->
Embedding
->
Vector Store

In [ ]:
from src.vector_store.vector_store import FaissVectorStore

In [ ]:
# Configure FaissVectorStore
dimension = embeddings.shape[1]

vectorStore = FaissVectorStore(dimension=dimension)

In [ ]:
# Adding Embeddings Into the Vector Store
vectorStore.add(embeddings=embeddings)

In [ ]:
# Checking total Indexes
vectorStore.total_vectors()

# Phase 2: Query and Response Generation

The second phase begins when a user submits a question.

The query is transformed into the same vector space as the document chunks so that semantically relevant information can be retrieved.

The pipeline is:

User Query
->
Query Embedding
->
Vector Search
->
Relevant Context
->
Prompt Construction
->
LLM Response Generation

## 1. User Query

The user provides a natural-language question.

The query represents the information that the user wants to retrieve from the indexed document.

In [ ]:
# query = "What is qualitative data analysis?"
query = "Steps to effective quantitative data analysis "


## 2. Query Embedding

The user query is converted into an embedding using the same embedding model used for the document chunks.

Using the same embedding model is important because both the query and the document chunks must exist in the same vector space for meaningful similarity comparison.

In [ ]:
from src.embeddings.embedder import chunk_encoder

In [ ]:
query_embedding = chunk_encoder([query])

## 3. Similarity Search

The query embedding is passed to the vector store.

The vector store compares the query vector with the stored document vectors and retrieves the most semantically similar chunks.

The result of this stage is the relevant context required to answer the user's question.

In [ ]:
# Querying the Vector store
distance, indices = vectorStore.search(query_embedding, 3)

In [ ]:
# Preparing the Retrieved text List
retrieved_text = []
for idx in indices[0]:
    retrieved_text.append(chunks[idx])

retrieved_text

## 4. Prompt Construction and Response Generation

The retrieved document context and the original user query are combined to construct a prompt for the Large Language Model.

The LLM uses the retrieved context to generate the final response.

The final stage is:

User Query
        +
Retrieved Context
        ->
Prompt
        ->
Large Language Model
        ->
Generated Response

In [ ]:
from src.generation.llm import LLMResponseGenerator

In [ ]:
# LLM Configuration
llm =  LLMResponseGenerator()

In [ ]:
# Prompt Building
prompt = llm.prompt_builder(context=retrieved_text, query=query)

In [ ]:
# Response Generation
response = llm.generate_response(prompt=prompt)

In [ ]:
print(response.text)